In [0]:
employee_df = spark.read.csv(path='/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv',header=True,inferSchema=True,sep='|',quote="'")
employee_df.display()

##Selecting Columns

###Using string

In [0]:
employee_df.select("id","name").display()

###Using Col Object

In [0]:
from pyspark.sql.functions import col
employee_df.select(col("id"),col("name").alias("employee_name")).display()

In [0]:
employee_df.columns

### Exclude column 

####Imperative style

In [0]:
columns_to_exclude = ['dob']
columns_to_include = []
for column in employee_df.columns:
  if column not in columns_to_exclude:
    columns_to_include.append(column)
employee_df.select(*columns_to_include).display()


####Functional Style

In [0]:
columns_to_exclude = ["dob"]
columns_to_include = list(
    filter(lambda x: x not in columns_to_exclude, employee_df.columns)
)
employee_df.select(*columns_to_include).display()

####Composition style

In [0]:
columns_to_exclude = ['dob','id']
columns_to_include = [col for col in employee_df.columns if col not in columns_to_exclude]
employee_df.select(*columns_to_include).display()

###Filter operations

In [0]:
employee_df.filter(col("gen")=="M").select("name").display()

In [0]:
employee_df.filter((col("gen")=="M") & (col("exp")>= 2)).select("name").display()

In [0]:
from pyspark.sql.functions import lower
employee_df.filter((lower("company") == "cisco")).select("name").display()

In [0]:
employee_df.filter(col("desig").isin(['Team Lead', 'Developer'])).display()

In [0]:
employee_df.groupBy('gen').count().orderBy(col('count'), ascending = False).display()

In [0]:
from pyspark.sql.functions import when 
employee_df.withColumn("exp_level", when(col("exp") >=10, "Senior").when(col("exp") >= 5, "Junior").when(col("exp")>=0,"Junior").otherwise("Invalid Experience ")).select("name",'exp',"exp_level").display()


In [0]:
employee_df.selectExpr(
    "*",
    "CASE WHEN exp >= 10 THEN 'Senior' WHEN exp >= 5 THEN 'Junior' WHEN exp >= 0 THEN 'Junior' ELSE 'Invalid Experience ' END AS exp_level",
).display()

In [0]:
employee_df.createOrReplaceTempView("Employee_vw")

In [0]:
%sql
select name, exp, CASE WHEN exp >= 10 THEN 'Senior' WHEN exp >= 5 THEN 'Junior' WHEN exp >= 0 THEN 'Junior' ELSE 'Invalid Experience ' END AS exp_level from Employee_vw;